# TODO

TODO

In [6]:
import os
import subprocess
import shutil
from datetime import datetime

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
import chromadb

embeddings_model_name = "sentence-transformers/all-distilroberta-v1"
chat_model_name = "Qwen/Qwen2.5-14B-Instruct"

PRJ_DIR = os.path.abspath("")
RAW_DIR = os.path.join(PRJ_DIR, "data", "raw")
DOC_DIR = os.path.join(PRJ_DIR, "data", "split")

In [7]:
# import the text splitting implementation
%run rag_split_text.ipynb

In [8]:

print("Checking for documents...")

subprocess.run("./pull_data.sh")

print("Splitting documents...")

shutil.rmtree(DOC_DIR, ignore_errors=True)
split_source_text_in_dir(RAW_DIR, DOC_DIR)

print(f"Loading the sentence transformer, {embeddings_model_name}...")

embeddings_model = SentenceTransformer(embeddings_model_name)

print("Loading ChromaDB and the documents...")

chroma = chromadb.PersistentClient(path=os.path.join(PRJ_DIR, "data", "chroma"))

vector_collection = chroma.get_or_create_collection(
    name="all-documents",
    # We generate the embeddings using sentence-transformers directly for pedagogical reasons
    embedding_function=None,
    metadata={
        "description": "my first Chroma collection",
        "created": str(datetime.now())
    },
    configuration={
        "hnsw": {
            # Cosine similarity is most useful for text embeddings I believe,
            # where scale is of little importance?
            "space": "cosine",
        }
    }
)

print("Generating embeddings and storing documents...")

paths = []
docs = []
for entry in os.scandir(DOC_DIR):
    if entry.is_file():
        paths.append(entry.path)

        with open(entry.path, "r") as f:
            docs.append(f.read())

embeddings = embeddings_model.encode(docs)

fnames = [os.path.basename(path) for path in paths]

vector_collection.upsert(
    ids=fnames,
    embeddings=embeddings,
    documents=docs,
    # metadatas=[{"chapter": 3, "verse": 16}, {"chapter": 3, "verse": 5}, {"chapter": 29, "verse": 11}, ...],
)

print("Done!")

Checking for documents...
Already up to date.
Extracting the markdown files
Copied: ./data/selection_round_github/tutorial1/README.md -> tutorial1_README.md
Copied: ./data/selection_round_github/tutorial4/README.md -> tutorial4_README.md
Copied: ./data/selection_round_github/tutorial2/README.md -> tutorial2_README.md
Copied: ./data/selection_round_github/README.md -> README.md
Copied: ./data/selection_round_github/tutorial3/README.md -> tutorial3_README.md
Splitting documents...
Loading the sentence transformer, sentence-transformers/all-distilroberta-v1...
Loading ChromaDB and the documents...
Generating embeddings and storing documents...
Done!
